In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Migration Demo: Ontology + Binding Pipeline

**Using the upstream `bigquery_ontology` package with separated ontology and binding YAML files.**

This notebook demonstrates the migrated V5 pipeline using `from_ontology_binding()` factory methods.
For the standard V5 demo using combined GraphSpec YAML, see
[`ontology_graph_v5_demo.ipynb`](ontology_graph_v5_demo.ipynb).


## Overview

This demo uses the **new upstream entry points** from the ontology migration:

- **Separated YAML files**: `*.ontology.yaml` (logical schema) + `*.binding.yaml` (physical mapping)
- **Upstream loaders**: `load_ontology()` + `load_binding()` from `bigquery_ontology`
- **Factory methods**: `from_ontology_binding()` on all runtime classes
- **Batch materialization**: `write_mode="batch_load"` with `TableStatus` reporting
- **Temporal lineage**: `LineageEdgeConfig` sidecar for cross-session evolution

The existing `load_graph_spec()` path still works unchanged. This notebook shows the new path.


## Install Dependencies


In [2]:
!pip install -q bigquery-agent-analytics google-cloud-bigquery pyyaml rdflib


DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621


ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement bigquery-agent-analytics (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /usr/local/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip
ERROR: No matching distribution found for bigquery-agent-analytics


## Authenticate & Configure


In [3]:
import os

try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab -- using default credentials.")

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
DATASET_ID = os.environ.get("BQ_DATASET", "agent_analytics")
TABLE_ID = os.environ.get("BQ_TABLE", "agent_events")

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Table    : {TABLE_ID}")


Not running in Colab -- using default credentials.
Project  : test-project-0728-467323
Dataset  : agent_analytics
Table    : agent_events


In [4]:
import uuid
from google.cloud import bigquery

RUN_ID = uuid.uuid4().hex[:8]
SCRATCH_DATASET = f"{DATASET_ID}_v5_{RUN_ID}"

bq = bigquery.Client(project=PROJECT_ID)
ds = bigquery.Dataset(f"{PROJECT_ID}.{SCRATCH_DATASET}")
ds.default_table_expiration_ms = 3600000
bq.create_dataset(ds, exists_ok=True)
print(f"Scratch dataset: {SCRATCH_DATASET}")
print(f"Tables auto-expire in 1 hour")


Scratch dataset: agent_analytics_v5_088d69eb
Tables auto-expire in 1 hour


## Step 1: Author Ontology + Binding YAML

The upstream `bigquery_ontology` package uses **two separate files**:
- `*.ontology.yaml` — logical schema (entities, relationships, properties, keys) with no physical binding info
- `*.binding.yaml` — maps the ontology to BigQuery tables and columns


In [5]:
ONTOLOGY_PATH = "ymgo.ontology.yaml"

_ONTOLOGY = '''\
ontology: YMGO_Context_Graph_V5
entities:
  - name: mako_DecisionPoint
    keys:
      primary: [decision_id]
    properties:
      - name: decision_id
        type: string
      - name: decision_type
        type: string

  - name: sup_YahooAdUnit
    keys:
      primary: [adUnitId]
    properties:
      - name: adUnitId
        type: string
      - name: adUnitName
        type: string
      - name: adUnitSize
        type: string
      - name: adUnitPosition
        type: string

  - name: mako_RejectionReason
    keys:
      primary: [rejection_id]
    properties:
      - name: rejection_id
        type: string
      - name: rejectionType
        type: string
      - name: rejectionRule
        type: string

relationships:
  - name: CandidateEdge
    from: mako_DecisionPoint
    to: sup_YahooAdUnit
    properties:
      - name: edge_type
        type: string
      - name: mako_scoreValue
        type: double

  - name: ForCandidate
    from: mako_RejectionReason
    to: sup_YahooAdUnit

  - name: sup_YahooAdUnitEvolvedFrom
    from: sup_YahooAdUnit
    to: sup_YahooAdUnit
    properties:
      - name: from_session_id
        type: string
      - name: to_session_id
        type: string
      - name: event_time
        type: timestamp
      - name: changed_properties
        type: string
'''

with open(ONTOLOGY_PATH, "w") as f:
    f.write(_ONTOLOGY)
print(f"Wrote {ONTOLOGY_PATH}")


Wrote ymgo.ontology.yaml


In [6]:
BINDING_PATH = "ymgo-bq.binding.yaml"

_BINDING = f'''\
binding: ymgo_bq
ontology: YMGO_Context_Graph_V5
target:
  backend: bigquery
  project: {PROJECT_ID}
  dataset: {SCRATCH_DATASET}
entities:
  - name: mako_DecisionPoint
    source: decision_points
    properties:
      - name: decision_id
        column: decision_id
      - name: decision_type
        column: decision_type
  - name: sup_YahooAdUnit
    source: yahoo_ad_units
    properties:
      - name: adUnitId
        column: adUnitId
      - name: adUnitName
        column: adUnitName
      - name: adUnitSize
        column: adUnitSize
      - name: adUnitPosition
        column: adUnitPosition
  - name: mako_RejectionReason
    source: rejection_reasons
    properties:
      - name: rejection_id
        column: rejection_id
      - name: rejectionType
        column: rejectionType
      - name: rejectionRule
        column: rejectionRule
relationships:
  - name: CandidateEdge
    source: candidate_edges
    from_columns: [decision_id]
    to_columns: [adUnitId]
    properties:
      - name: edge_type
        column: edge_type
      - name: mako_scoreValue
        column: mako_scoreValue
  - name: ForCandidate
    source: rejection_mappings
    from_columns: [rejection_id]
    to_columns: [adUnitId]
  - name: sup_YahooAdUnitEvolvedFrom
    source: sup_yahoo_ad_unit_lineage
    from_columns: [adUnitId]
    to_columns: [adUnitId]
    properties:
      - name: from_session_id
        column: from_session_id
      - name: to_session_id
        column: to_session_id
      - name: event_time
        column: event_time
      - name: changed_properties
        column: changed_properties
'''

with open(BINDING_PATH, "w") as f:
    f.write(_BINDING)
print(f"Wrote {BINDING_PATH}")


Wrote ymgo-bq.binding.yaml


In [7]:
from bigquery_ontology import load_ontology, load_binding

ontology = load_ontology(ONTOLOGY_PATH)
binding = load_binding(BINDING_PATH, ontology=ontology)

print(f"Ontology: {ontology.ontology} ({len(ontology.entities)} entities, {len(ontology.relationships)} rels)")
print(f"Binding: {binding.binding} (target: {binding.target.project}.{binding.target.dataset})")


Ontology: YMGO_Context_Graph_V5 (3 entities, 3 rels)
Binding: ymgo_bq (target: test-project-0728-467323.agent_analytics_v5_088d69eb)


In [8]:
!gm validate {ONTOLOGY_PATH}
!gm validate {BINDING_PATH}


zsh:1: command not found: gm


zsh:1: command not found: gm


## Step 2: Create Runtime Classes via `from_ontology_binding()`

The SDK runtime classes now accept `Ontology` + `Binding` directly via factory methods.
The materializer and compiler derive `project_id`/`dataset_id` from `binding.target`.
The graph manager takes explicit telemetry project/dataset.


In [9]:
from bigquery_agent_analytics.runtime_spec import LineageEdgeConfig

lineage_config = {
    "sup_YahooAdUnitEvolvedFrom": LineageEdgeConfig(
        from_session_column="from_session_id",
        to_session_column="to_session_id",
    ),
}
print(f"Lineage config: {list(lineage_config.keys())}")


Lineage config: ['sup_YahooAdUnitEvolvedFrom']


/Users/haiyuancao/adk-python/src/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [10]:
from bigquery_agent_analytics.ontology_graph import OntologyGraphManager

mgr = OntologyGraphManager.from_ontology_binding(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    ontology=ontology,
    binding=binding,
    lineage_config=lineage_config,
    table_id=TABLE_ID,
    endpoint="gemini-2.5-flash",
)
print(f"Spec: {mgr.spec.name}")
print(f"Entities: {[e.name for e in mgr.spec.entities]}")
print(f"Relationships: {[r.name for r in mgr.spec.relationships]}")


Spec: YMGO_Context_Graph_V5
Entities: ['mako_DecisionPoint', 'sup_YahooAdUnit', 'mako_RejectionReason']
Relationships: ['CandidateEdge', 'ForCandidate', 'sup_YahooAdUnitEvolvedFrom']


In [11]:
SESSION_IDS = ["adcp-033c95d7a97d"]

graph = mgr.extract_graph(session_ids=SESSION_IDS, use_ai_generate=True)
print(f"Extracted {len(graph.nodes)} nodes, {len(graph.edges)} edges")
for node in graph.nodes[:5]:
    print(f"  [{node.entity_name}] {node.node_id}")


Extracted 4 nodes, 2 edges
  [sup_YahooAdUnit] adcp-033c95d7a97d:sup_YahooAdUnit:adUnitId=Yahoo Homepage_display_Premium_Placement
  [sup_YahooAdUnit] adcp-033c95d7a97d:sup_YahooAdUnit:adUnitId=Yahoo Mail_display_Standard
  [mako_DecisionPoint] adcp-033c95d7a97d:mako_DecisionPoint:decision_id=allocate_media_budget_1
  [mako_DecisionPoint] adcp-033c95d7a97d:mako_DecisionPoint:decision_id=provision_campaign_in_gam_1


## Step 3: Materialize with `batch_load`

The materializer derives project/dataset from `binding.target` automatically.


In [12]:
from bigquery_agent_analytics.ontology_materializer import OntologyMaterializer

materializer = OntologyMaterializer.from_ontology_binding(
    ontology=ontology,
    binding=binding,
    lineage_config=lineage_config,
    write_mode="batch_load",
)

# Create tables first (batch_load needs existing tables).
tables = materializer.create_tables()
print("Tables created:")
for name, ref in tables.items():
    print(f"  {name} -> {ref}")

result = materializer.materialize_with_status(graph, SESSION_IDS)
print(f"\nRows: {result.row_counts}")
for ref, ts in result.table_statuses.items():
    print(f"  {ref}: cleanup={ts.cleanup_status} insert={ts.insert_status} idempotent={ts.idempotent}")


Tables created:
  mako_DecisionPoint -> test-project-0728-467323.agent_analytics_v5_088d69eb.decision_points
  sup_YahooAdUnit -> test-project-0728-467323.agent_analytics_v5_088d69eb.yahoo_ad_units
  mako_RejectionReason -> test-project-0728-467323.agent_analytics_v5_088d69eb.rejection_reasons
  CandidateEdge -> test-project-0728-467323.agent_analytics_v5_088d69eb.candidate_edges
  ForCandidate -> test-project-0728-467323.agent_analytics_v5_088d69eb.rejection_mappings
  sup_YahooAdUnitEvolvedFrom -> test-project-0728-467323.agent_analytics_v5_088d69eb.sup_yahoo_ad_unit_lineage



Rows: {'sup_YahooAdUnit': 2, 'mako_DecisionPoint': 2, 'CandidateEdge': 2}
  test-project-0728-467323.agent_analytics_v5_088d69eb.yahoo_ad_units: cleanup=deleted insert=inserted idempotent=True
  test-project-0728-467323.agent_analytics_v5_088d69eb.decision_points: cleanup=deleted insert=inserted idempotent=True
  test-project-0728-467323.agent_analytics_v5_088d69eb.candidate_edges: cleanup=deleted insert=inserted idempotent=True


## Step 4: Property Graph DDL


In [13]:
from bigquery_agent_analytics.ontology_property_graph import OntologyPropertyGraphCompiler

compiler = OntologyPropertyGraphCompiler.from_ontology_binding(
    ontology=ontology,
    binding=binding,
    lineage_config=lineage_config,
)
print(compiler.get_ddl())


CREATE OR REPLACE PROPERTY GRAPH `test-project-0728-467323.agent_analytics_v5_088d69eb.YMGO_Context_Graph_V5`
  NODE TABLES (
    `test-project-0728-467323.agent_analytics_v5_088d69eb.decision_points` AS mako_DecisionPoint
      KEY (decision_id, session_id)
      LABEL mako_DecisionPoint
      PROPERTIES (
        decision_id,
        decision_type,
        session_id,
        extracted_at
      ),
    `test-project-0728-467323.agent_analytics_v5_088d69eb.yahoo_ad_units` AS sup_YahooAdUnit
      KEY (adUnitId, session_id)
      LABEL sup_YahooAdUnit
      PROPERTIES (
        adUnitId,
        adUnitName,
        adUnitSize,
        adUnitPosition,
        session_id,
        extracted_at
      ),
    `test-project-0728-467323.agent_analytics_v5_088d69eb.rejection_reasons` AS mako_RejectionReason
      KEY (rejection_id, session_id)
      LABEL mako_RejectionReason
      PROPERTIES (
        rejection_id,
        rejectionType,
        rejectionRule,
        session_id,
        extrac

In [14]:
created = compiler.create_property_graph()
print(f"Property Graph created: {created}")


Property Graph created: True


## Step 5: Temporal Lineage


### Demo Note: Synthetic Lineage Input

The lineage walkthrough below uses a synthetic follow-up session that reuses a known `adUnitId` with modified properties. This proves the end-to-end lineage path without depending on real telemetry sessions having overlapping entity keys.


In [15]:
from bigquery_agent_analytics.ontology_models import (
    ExtractedGraph, ExtractedNode, ExtractedProperty,
)

session_a_ad_units = [n for n in graph.nodes if n.entity_name == "sup_YahooAdUnit"]
if session_a_ad_units:
    original = session_a_ad_units[0]
    original_props = {p.name: p.value for p in original.properties}
    shared_id = original_props.get("adUnitId", "unknown")

    synthetic_node = ExtractedNode(
        node_id=f"sess-synthetic-B:sup_YahooAdUnit:adUnitId={shared_id}",
        entity_name="sup_YahooAdUnit",
        labels=["sup_YahooAdUnit"],
        properties=[
            ExtractedProperty(name="adUnitId", value=shared_id),
            ExtractedProperty(name="adUnitName", value=original_props.get("adUnitName", "") + " (Redesigned)"),
            ExtractedProperty(name="adUnitSize", value=original_props.get("adUnitSize", "300x250")),
            ExtractedProperty(name="adUnitPosition", value="BTF"),
        ],
    )
    synthetic_graph = ExtractedGraph(name=mgr.spec.name, nodes=[synthetic_node], edges=[])
    result_b = materializer.materialize_with_status(synthetic_graph, ["sess-synthetic-B"])
    print(f"Synthetic Session B: {result_b.row_counts}")
else:
    print("No ad units in Session A")
    synthetic_graph = ExtractedGraph(name=mgr.spec.name, nodes=[], edges=[])


Synthetic Session B: {'sup_YahooAdUnit': 1}


In [16]:
from bigquery_agent_analytics.ontology_graph import detect_lineage_edges

lineage_edges = detect_lineage_edges(
    current_graph=synthetic_graph,
    current_session_id="sess-synthetic-B",
    prior_graphs={SESSION_IDS[0]: graph},
    lineage_entity_types=["sup_YahooAdUnit"],
    spec=mgr.spec,
)
print(f"Lineage edges detected: {len(lineage_edges)}")
for edge in lineage_edges:
    print(f"  {edge.relationship_name}")
    for p in edge.properties:
        print(f"    {p.name} = {p.value}")


Lineage edges detected: 1
  sup_YahooAdUnitEvolvedFrom
    from_session_id = adcp-033c95d7a97d
    to_session_id = sess-synthetic-B
    event_time = 2026-04-15T07:57:23.396466+00:00
    changed_properties = adUnitName,adUnitPosition


In [17]:
lineage_graph = ExtractedGraph(name=mgr.spec.name, nodes=[], edges=lineage_edges)
lineage_result = materializer.materialize_with_status(lineage_graph, ["sess-synthetic-B"])
print(f"Lineage rows: {lineage_result.row_counts}")
for ref, ts in lineage_result.table_statuses.items():
    print(f"  {ref}: idempotent={ts.idempotent}")


Lineage rows: {'sup_YahooAdUnitEvolvedFrom': 1}
  test-project-0728-467323.agent_analytics_v5_088d69eb.sup_yahoo_ad_unit_lineage: idempotent=True


## Step 6: GQL Queries


In [18]:
from bigquery_agent_analytics.ontology_orchestrator import compile_showcase_gql, compile_lineage_gql

showcase_gql = compile_showcase_gql(mgr.spec, PROJECT_ID, SCRATCH_DATASET)
print("=== Forward Traversal GQL ===")
print(showcase_gql)

lineage_gql = compile_lineage_gql(
    spec=mgr.spec,
    project_id=PROJECT_ID,
    dataset_id=SCRATCH_DATASET,
    relationship_name="sup_YahooAdUnitEvolvedFrom",
)
print("\n=== Lineage GQL ===")
print(lineage_gql)


=== Forward Traversal GQL ===
GRAPH `test-project-0728-467323.agent_analytics_v5_088d69eb.YMGO_Context_Graph_V5`
MATCH
  (dp:mako_DecisionPoint)-[ece:CandidateEdge]->(yau:sup_YahooAdUnit)
WHERE dp.session_id = @session_id
RETURN
  dp.decision_id AS src_decision_id,
  dp.decision_type AS src_decision_type,
  ece.edge_type,
  ece.mako_scoreValue,
  yau.adUnitId AS dst_adUnitId,
  yau.adUnitName AS dst_adUnitName,
  yau.adUnitSize AS dst_adUnitSize,
  yau.adUnitPosition AS dst_adUnitPosition
ORDER BY dp.decision_id
LIMIT @result_limit


=== Lineage GQL ===
GRAPH `test-project-0728-467323.agent_analytics_v5_088d69eb.YMGO_Context_Graph_V5`
MATCH
  (prev:sup_YahooAdUnit)-[eyauef:sup_YahooAdUnitEvolvedFrom]->(cur:sup_YahooAdUnit)
WHERE cur.session_id = @session_id
RETURN
  prev.adUnitId AS prior_adUnitId,
  prev.adUnitName AS prior_adUnitName,
  prev.adUnitSize AS prior_adUnitSize,
  prev.adUnitPosition AS prior_adUnitPosition,
  eyauef.from_session_id,
  eyauef.to_session_id,
  eyauef.event_

In [19]:
job = bq.query(
    showcase_gql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("session_id", "STRING", SESSION_IDS[0]),
            bigquery.ScalarQueryParameter("result_limit", "INT64", 50),
        ]
    ),
)
results = job.to_dataframe()
print(f"Forward traversal: {len(results)} rows")
results.head(10)


Forward traversal: 2 rows


,src_decision_id,src_decision_type,edge_type,mako_scoreValue,dst_adUnitId,dst_adUnitName,dst_adUnitSize,dst_adUnitPosition
0,allocate_media_budget_1,allocate_media_budget,allocation,9424.81,Yahoo Homepage_display_Premium_Placement,Yahoo Homepage,display,Premium Placement
1,allocate_media_budget_1,allocate_media_budget,allocation,40575.19,Yahoo Mail_display_Standard,Yahoo Mail,display,Standard


In [20]:
try:
    job = bq.query(
        lineage_gql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("session_id", "STRING", "sess-synthetic-B"),
                bigquery.ScalarQueryParameter("result_limit", "INT64", 50),
            ]
        ),
    )
    lineage_results = job.to_dataframe()
    print(f"Lineage traversal: {len(lineage_results)} rows")
    if len(lineage_results) > 0:
        print(lineage_results.head(10))
except Exception as e:
    print(f"Lineage query failed: {e}")


Lineage traversal: 1 rows
                             prior_adUnitId prior_adUnitName prior_adUnitSize  \
0  Yahoo Homepage_display_Premium_Placement   Yahoo Homepage          display   

  prior_adUnitPosition    from_session_id     to_session_id  \
0    Premium Placement  adcp-033c95d7a97d  sess-synthetic-B   

                        event_time         changed_properties  \
0 2026-04-15 07:57:23.396466+00:00  adUnitName,adUnitPosition   

                           current_adUnitId           current_adUnitName  \
0  Yahoo Homepage_display_Premium_Placement  Yahoo Homepage (Redesigned)   

  current_adUnitSize current_adUnitPosition  
0            display                    BTF  


In [21]:
bq.delete_dataset(f"{PROJECT_ID}.{SCRATCH_DATASET}", delete_contents=True, not_found_ok=True)
print(f"Deleted scratch dataset: {SCRATCH_DATASET}")


Deleted scratch dataset: agent_analytics_v5_088d69eb


### Operational Note: AI Extraction Variance

The unstructured extraction relies on `AI.GENERATE`. The demo is stable because it uses controlled fixtures, isolated scratch targets, and schema filtering drops extra AI-emitted fields.


## Summary

This notebook demonstrated the **migrated V5 pipeline** using upstream `bigquery_ontology` entry points:

| What | Old path | New path |
|------|----------|----------|
| **Spec format** | Combined `GraphSpec` YAML | Separated `*.ontology.yaml` + `*.binding.yaml` |
| **Loading** | `load_graph_spec(path, env=...)` | `load_ontology()` + `load_binding()` |
| **Validation** | SDK-only | `gm validate` CLI + upstream cross-validation |
| **Runtime** | `OntologyGraphManager(spec=...)` | `OntologyGraphManager.from_ontology_binding(...)` |
| **Materialization** | `OntologyMaterializer(project, dataset, spec)` | `OntologyMaterializer.from_ontology_binding(ontology, binding)` |
| **DDL** | `OntologyPropertyGraphCompiler(project, dataset, spec)` | `OntologyPropertyGraphCompiler.from_ontology_binding(ontology, binding)` |
| **Lineage** | `BindingSpec.from_session_column` | `LineageEdgeConfig` sidecar |

Both paths coexist. The existing `load_graph_spec()` path is unchanged.

This notebook is a deterministic demo, not a proof that real-world telemetry always yields lineage overlap or perfectly stable AI extraction.
